In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace=underlying_embeddings.model
)


/Users/dohyunkim/Documents/langchain-for-ai-agent/.venv/lib/python3.13/site-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [2]:
from langchain_chroma import Chroma

CHROMA_PATH = "./chroma_db"


db = Chroma(
    collection_name="rag_collection",
    embedding_function=cached_embedder,
    persist_directory=CHROMA_PATH,
    collection_metadata={"hnsw:space": "cosine"}  # 옵션: "cosine", "l2" (유클리드), "ip" (내적)
)


In [3]:
# 특징: documents= 인자가 없습니다. 
# 즉, 새로운 문서를 추가하는 게 아니라 기존에 저장된 데이터를 검색하기 위한 용도로 주로 사용합니다
db = Chroma(
    collection_name="rag_collection",
    embedding_function=cached_embedder,
    #  "텍스트를 숫자로 바꿀 때(임베딩), 이미 계산한 적이 있는 내용은 다시 계산하지 않고 저장해둔 값을 꺼내 쓰겠다"는 의미입니다.

    persist_directory=CHROMA_PATH,
    # 1. 정보가 저장된 위치 (읽기)
    # 이미 CHROMA_PATH 경로에 벡터 데이터(임베딩된 문서들)가 저장되어 있다면, 새로 데이터를 생성하지 않고 그 폴더에 있는 기존 데이터를 불러와서 사용하겠다는 뜻입니다.
    
    # 2. 정보가 저장될 위치 (쓰기)
    # 만약 해당 경로에 데이터가 없다면, 현재 처리 중인 문서들을 임베딩하여 그 폴더 안에 파일 형태로 영구히 저장(Persist)하겠다는 뜻입니다.

    collection_metadata={"hnsw:space": "cosine"}  # 옵션: "cosine", "l2" (유클리드), "ip" (내적)

    # hnsw: ChromaDB가 내부적으로 사용하는 고속 검색 알고리즘(Hierarchical Navigable Small World)의 약자입니다.
    # space: 검색 공간에서 거리를 측정하는 방식을 뜻합니다.
    # 즉, 이 설정은 "데이터를 찾을 때 코사인 유사도 알고리즘을 사용하여 가장 의미가 가까운 문장을 찾아라"라는 지시입니다.
)

In [4]:

query = "Tesla 투자 비중이 얼마나 되나요?"
results = db.similarity_search(query)
# 벡터스토어에 있는 문서와 유사도를 계산하고, 유사도가 높은 순서대로 검색 결과를 반환한다

print(len(results))

print(f"검색된 문서 내용:\n{results[0].page_content}")
# 가장 유사한 문서의 내용을 반환한다
# 이 내용을 LLM에게 전달해서 답변을 생성할 수 있다
# 이것이 RAG 





4
검색된 문서 내용:
Holdings Data - ARKK
As of 11/26/2025
ARKK
ARK Innovation ETF
Company Ticker CUSIP Shares Market Value ($) Weight (%)
1 TESLA INC TSLA 88160R101 2,204,438 $924,541,297.20 12.26%
2 TEMPUS AI INC TEM 88023B103 5,465,331 $419,956,034.04 5.57%
3 ROKU INC ROKU 77543R102 4,347,025 $412,445,732.00 5.47%


In [5]:
# 검색기 생성
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.4}
    # score_threshold: 유사도 점수가 0.4 이상인 문서만 검색하겠다는 의미
    # 따라서 유사도 점수를 높이면 검색 결과가 없을 수도 있다.
)

results = retriever.invoke(query)

print(len(results))

print(results[0].page_content)


No relevant docs were retrieved using the relevance score threshold 0.4


0


IndexError: list index out of range

In [6]:
# 검색기 생성
retriever2 = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3,}

)


results2 = retriever2.invoke(query)

print(len(results2))

print(results2[0].page_content)

3
Holdings Data - ARKK
As of 11/26/2025
ARKK
ARK Innovation ETF
Company Ticker CUSIP Shares Market Value ($) Weight (%)
1 TESLA INC TSLA 88160R101 2,204,438 $924,541,297.20 12.26%
2 TEMPUS AI INC TEM 88023B103 5,465,331 $419,956,034.04 5.57%
3 ROKU INC ROKU 77543R102 4,347,025 $412,445,732.00 5.47%
